임성열 Ph.D. 2025. 02.11, SKALA 수업 학습목적으로만 활용 바랍니다.

In [15]:
# !pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
# !pip install openai==1.55.3 httpx==0.27.2

In [16]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [17]:
from crewai import Agent, Task, Crew

In [18]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

api_key= "################################" # 개인 API 키를 입력합니다.
os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o'

#hi

Agent(에이전트) 생성하기

Agent를 정의하고, role(역할), goal(목표), backstory(배경 설명)를 제공합니다.

LLM(대규모 언어 모델)은 롤플레잉을 할 때 더 나은 성능을 보이는 것으로 확인되었습니다.

### Agent: 기획자(Planner)

**참고**: _여러 문자열_ 사용의 이점:
```Python
varname = "텍스트의 첫 번째 줄"
          "텍스트의 두 번째 줄"
```

_삼중 따옴표 독스트링_ 사용과 비교했을 때:
```Python
varname = """텍스트의 첫 번째 줄
             텍스트의 두 번째 줄
          """
```
여러 문자열을 사용하면 공백과 줄바꿈 문자가 추가되는 것을 피할 수 있어, LLM에 전달할 때 더 나은 형식을 유지할 수 있습니다.

In [19]:
planner = Agent(
    role="콘텐츠 기획자",
    goal="{topic}에 대한 흥미롭고 사실에 기반한 콘텐츠 기획",
    backstory="당신은 {topic}에 대한 블로그 글을 "
              "기획하는 일을 하고 있습니다. "
              "독자들이 새로운 것을 배우고 "
              "정보에 기반한 결정을 내릴 수 있도록 "
              "도움이 되는 정보를 수집합니다. "
              "당신의 작업은 콘텐츠 작성자가 "
              "이 주제로 글을 쓸 수 있는 토대가 됩니다."
              "결과물은 한국어로 생성될 예정입니다.",
    allow_delegation=False,
    verbose=True
)

### Agent: Writer

In [20]:
writer = Agent(
    role="콘텐츠 작성자",
    goal="{topic}에 대한 통찰력 있고 사실에 기반한 "
         "의견 기사 작성",
    backstory="당신은 {topic}에 대한 새로운 "
              "의견 기사를 작성하고 있습니다. "
              "콘텐츠 기획자가 제공한 개요와 "
              "주제 관련 맥락을 바탕으로 글을 씁니다. "
              "콘텐츠 기획자가 제시한 "
              "주요 목표와 방향을 따릅니다. "
              "또한 객관적이고 공정한 통찰을 제공하고 "
              "이를 콘텐츠 기획자가 제공한 "
              "정보로 뒷받침합니다. "
              "의견 기사에서 객관적 진술과 "
              "구별되는 개인적 의견을 명시합니다."
              "결과물은 한국어로 생성될 예정입니다.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [21]:
editor = Agent(
    role="편집자",
    goal="주어진 블로그 글을 조직의 글쓰기 스타일에 "
         "맞게 편집",
    backstory="당신은 콘텐츠 작성자로부터 "
              "블로그 글을 받는 편집자입니다. "
              "블로그 글을 검토하여 저널리즘의 모범 사례를 "
              "따르고 있는지 확인하고, "
              "의견이나 주장을 제시할 때 "
              "균형 잡힌 관점을 제공하며, "
              "가능한 한 주요 논쟁적 주제나 "
              "의견을 피하도록 하는 것이 당신의 목표입니다."
              "결과물은 한국어로 생성될 예정입니다.",
    allow_delegation=False,
    verbose=True
)

## Task(작업) 생성하기       #Task 상세히 작업하기!

- Task를 정의하고, `description`(설명), `expected_output`(예상 결과물), `agent`(수행 에이전트)를 제공합니다.


### Task: Plan

In [22]:
plan = Task(
    description=(
        "1. {topic}에 대한 최신 트렌드, 주요 인물, "
            "그리고 주목할 만한 뉴스를 우선순위화하기.\n"
        "2. 목표 독자층의 관심사와 pain point를 "
            "고려하여 파악하기.\n"
        "3. 서론, 핵심 요점, 행동 유도(CTA)를 포함한 "
            "상세한 콘텐츠 개요 개발하기.\n"
        "4. SEO 키워드와 관련 데이터 또는 출처 포함하기."
    ),
    expected_output="개요, 독자층 분석, "
        "SEO 키워드, 참고 자료를 포함한 "
        "포괄적인 콘텐츠 기획 문서."
        "결과물은 한국어로 생성될 예정입니다.",
    agent=planner,
)

### Task: Write

In [23]:
write = Task(
    description=(
        "1. 콘텐츠 기획을 활용하여 {topic}에 대한 "
            "매력적인 블로그 글 작성하기.\n"
        "2. SEO 키워드를 자연스럽게 포함하기.\n"
        "3. 섹션/부제목을 흥미롭게 적절한 "
            "이름으로 지정하기.\n"
        "4. 매력적인 서론, 통찰력 있는 본문, "
            "요약하는 결론으로 글을 구성하기.\n"
        "5. 문법 오류를 점검하고 "
            "브랜드의 톤앤매너와 일치하는지 확인하기.\n"
    ),
    expected_output="각 섹션이 2~3개의 문단으로 구성된, "
        "출판 준비가 완료된 마크다운 형식의 "
        "잘 작성된 블로그 글."
        "결과물은 한국어로 생성될 예정입니다.",
    agent=writer,
)

### Task: Edit

In [24]:
edit = Task(
    description=("주어진 블로그 글의 문법 오류를 검토하고 "
                 "브랜드의 톤앤매너와 일치하는지 확인하기."),
    expected_output="각 섹션이 2~3개의 문단으로 구성된, "
                    "출판 준비가 완료된 마크다운 형식의 "
                    "잘 작성된 블로그 글."
                    "결과물은 한국어로 생성될 예정입니다.",
    agent=editor
)

## Crew(팀) 생성하기

- Agent들로 구성된 팀을 생성합니다
- 해당 Agent들이 수행할 작업들을 전달합니다.
    - **참고**: *이 간단한 예시에서는* 작업들이 순차적으로 수행됩니다(즉, 서로 의존적임). 따라서 목록에서의 작업 _순서_가 _중요_합니다.
- `verbose=2`를 설정하면 실행의 모든 로그를 확인할 수 있습니다.


이 설정에서:
1. `planner`가 먼저 콘텐츠를 기획합니다
2. `writer`가 기획된 내용을 바탕으로 글을 작성합니다
3. `editor`가 최종적으로 작성된 글을 검토하고 편집합니다

In [25]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

2025-02-12 17:03:30,145 - 139703672033280 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


## Running the Crew

In [26]:
result = crew.kickoff(inputs={"topic": "LLM(Large Language Model)을 이용한 지능형 에이전트 경쟁력 제고 방안"})

 [DEBUG]: == Working Agent: 콘텐츠 기획자
 [INFO]: == Starting Task: 1. LLM(Large Language Model)을 이용한 지능형 에이전트 경쟁력 제고 방안에 대한 최신 트렌드, 주요 인물, 그리고 주목할 만한 뉴스를 우선순위화하기.
2. 목표 독자층의 관심사와 pain point를 고려하여 파악하기.
3. 서론, 핵심 요점, 행동 유도(CTA)를 포함한 상세한 콘텐츠 개요 개발하기.
4. SEO 키워드와 관련 데이터 또는 출처 포함하기.


> Entering new CrewAgentExecutor chain...
Thought: I now can give a great answer
Final Answer: 

## 콘텐츠 기획: LLM(Large Language Model)을 이용한 지능형 에이전트 경쟁력 제고 방안

### 개요

1. **서론**
   - LLM(Large Language Model)의 정의 및 중요성
   - 지능형 에이전트에서 LLM의 역할
   - 이 블로그의 목적: LLM을 활용한 지능형 에이전트의 경쟁력 제고 방법 소개

2. **핵심 요점**
   - **최신 트렌드**
     - 자연어 처리(NLP)와 AI의 발전
     - LLM의 대화형 AI 에이전트 적용 사례
     - AI 에이전트 성능 향상을 위한 최신 기술
   - **주요 인물 및 기업**
     - OpenAI 및 Google AI의 기술 리더들
     - Microsoft와 NVIDIA의 협업 사례
   - **주목할 만한 뉴스**
     - 최근 AI 관련 컨퍼런스 및 발표 자료
     - AI 규제 및 윤리적 고려 사항
   - **실질적인 경쟁력 제고 방안**
     - AI 에이전트의 사용자 경험 개선
     - 개인화 및 맞춤형 서비스 제공
     - 데이터 보안 및 프라이버시 강화

3. **행동 유도(CTA)**
   - 독자들이 LLM을 활용한 지능형 에이전트 개발 시 고려해야

In [27]:
from IPython.display import Markdown
Markdown(result)

```markdown
# LLM을 활용한 지능형 에이전트 경쟁력 제고 방안

## 서론: LLM이 혁신하는 지능형 에이전트의 세계

최근 인공지능(AI) 분야에서 주목받고 있는 LLM(Large Language Model)은 자연어 처리(NLP) 기술의 혁신을 이끌고 있습니다. 이러한 모델은 방대한 데이터로부터 학습하여 인간과 유사한 수준의 언어 이해 및 생성 능력을 보여줘, 다양한 분야에서 활용되고 있습니다. 특히, 지능형 에이전트에서 LLM의 중요성은 더욱 부각되며, 이는 사용자와의 상호작용을 한층 더 인간적이고 자연스럽게 만들어 줍니다. 이 블로그의 목적은 LLM을 활용한 지능형 에이전트의 경쟁력 제고 방법을 소개하고, 이를 통해 AI 기술 전문가와 관련 산업 종사자들이 실질적인 비즈니스 이점을 얻을 수 있도록 돕는 것입니다.

## 핵심 요점

### 최신 트렌드: AI의 진화와 LLM의 역할

AI 기술은 날로 발전하고 있으며, 특히 자연어 처리(NLP) 분야에서 LLM의 등장은 대화형 AI 에이전트의 성능을 크게 향상시켰습니다. 예를 들어, OpenAI의 GPT 시리즈는 대화형 AI의 가능성을 넓혔으며, Google AI는 이를 바탕으로 다양한 응용 프로그램을 개발하고 있습니다. 이러한 최신 기술들은 AI 에이전트의 성능을 극대화하는 데 기여하고 있습니다. 앞으로 이러한 발전이 AI 에이전트의 사용자 경험을 더욱 개선할 것으로 기대됩니다.

### 주요 인물 및 기업: AI 혁신을 이끄는 리더들

OpenAI와 Google AI는 LLM 연구와 개발의 선두주자입니다. 이들 기업은 지속적인 연구와 개발을 통해 AI 기술의 경계를 확장하고 있습니다. 또한, Microsoft와 NVIDIA의 협업은 AI 기술 발전에 있어 중요한 사례로 꼽힙니다. 이들의 기술적 리더십은 지능형 에이전트의 개발에 있어 강력한 토대를 제공하고 있습니다.

### 주목할 만한 뉴스: AI의 규제와 윤리적 고려

최근 AI 관련 컨퍼런스와 발표 자료들은 AI의 윤리적 고려와 규제 필요성을 강조하고 있습니다. AI 기술이 발전함에 따라, 데이터 프라이버시와 보안 문제는 점점 더 중요한 이슈로 부각되고 있습니다. 이러한 배경 속에서, AI 기술자는 윤리적인 책임을 다하기 위해 지속적으로 노력해야 합니다.

### 실질적인 경쟁력 제고 방안

LLM을 활용한 지능형 에이전트의 경쟁력을 높이기 위해서는 몇 가지 실질적인 전략이 필요합니다. 첫째, AI 에이전트의 사용자 경험을 개선하여 더 개인화된 서비스를 제공할 수 있습니다. 둘째, 데이터 보안 및 프라이버시를 강화하여 사용자의 신뢰를 확보해야 합니다. 마지막으로, 맞춤형 서비스 개발을 통해 고객 만족도를 높일 수 있습니다.

## 결론: LLM을 활용한 AI 에이전트 개발의 미래

LLM을 활용한 지능형 에이전트는 AI 기술의 미래를 밝히는 중요한 열쇠입니다. AI 기술 전문가와 개발자들은 이러한 모델을 통해 더욱 혁신적이고 효율적인 솔루션을 개발할 수 있습니다. 독자 여러분께서는 LLM을 활용한 지능형 에이전트 개발 시, 최신 트렌드를 주시하고 윤리적 고려를 반영한 전략을 수립하시기를 권장합니다. 관련 학습 자료와 워크숍에 참여하여 끊임없이 변화하는 AI 기술 환경에 적응해 나가시길 바랍니다.
```

This content aligns with the journalistic integrity by providing a balanced perspective and avoiding controversy while being consistent with the organization's writing style.

- Display the results of your execution as markdown in the notebook.